In [1]:
%pip install shapiq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import openml
import pandas as pd

# Load from OpenML ID 1461 (task 7592)
d = openml.datasets.get_dataset("bank-marketing", version=1)
X, y, cat_mask, attr_names = d.get_data(target=d.default_target_attribute)
X = pd.DataFrame(X, columns=attr_names[:-1])
y = pd.Series(y, name=d.default_target_attribute)

In [3]:
import sys
import os
# In python file use __file__ , not notebook's directory 
notebook_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..', 'custom_packages', 'bonXAI-main')))

In [4]:
from bonXAI.core.tabularPreprocessor import TabularPreprocessor
prep = TabularPreprocessor(
    task_type="classification",
    id_like_threshold=0.99,
    scale_all_numeric_after_encoding=True,
    scale_target_in_regression=False,
    random_state=0,
)
X, y = prep.fit_transform(X, y)

In [5]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=2000, n_jobs=None) 
clf.fit(X, y)

LogisticRegression(max_iter=2000)

In [ ]:
from bonXAI.core.explainer import Explainer
exp = Explainer(model=clf, explainer_name="shapiq", strategy="kernel", seed=0)
vals, t = exp.explain(
    x_foreground=X.values[:10],
    x_background=X.values,
    n_jobs=1
)
print(vals.shape)           # -> (10, n_features); order 1 features - usual SHAP values
print(vals)
pairs = exp.shapiq_pairwise_  # list length 10; each item is a matrix or None
print(pairs[0].shape)        # -> (n_features, n_features) or None; order 2 features - SHAP interaction values
print(pairs)

Explaining with SHAPIQ (k-SII, max_order=2). 10 samples to explain using 45211 background samples.
(10, 15)
[[ 1.05252038e-03  1.01310963e-03  2.30652828e-03 -3.69727019e-02
   1.32358083e-02 -1.02953347e-02 -5.03174356e-03  6.07411375e-03
  -8.67697951e-03  1.35663148e-02  1.82448155e-04 -2.28648453e-02
   5.96011672e-03 -5.73683995e-02 -2.03466469e-02]
 [-4.32081347e-04 -1.40793134e-03  3.11702945e-03 -6.30279645e-02
   1.31911732e-02 -1.03204545e-02 -4.98667875e-03 -2.31258177e-03
   1.65372767e-02 -2.48362416e-03 -9.01150290e-04 -2.15995224e-02
   5.55062004e-03 -5.61772849e-02 -1.99225983e-02]
 [-1.68256099e-03  1.04113946e-04  3.79655080e-03 -7.23379677e-02
   1.25486787e-02 -1.03961108e-02 -4.39541997e-03 -9.60391700e-03
  -9.15671267e-03 -1.05204475e-03 -1.59561099e-03 -1.87296918e-02
  -3.18232072e-02 -5.03461265e-02 -1.94390576e-02]
 [-3.80565157e-04  6.65591468e-04  3.04329173e-03 -7.25096986e-02
   1.25569926e-02 -1.06894887e-02 -4.69985518e-03 -1.25014356e-02
  -8.71359761

### Pairs:
- A **list of length len(x_foreground)** (one per explained sample)
- **Each item** a **num(features) × num(features) symmetric matrix** of **order-2 (pairwise) interaction values**
- **Meaning** entry (i, j) is the **extra effect of features i and j together** beyond the sum of their individual main effects